## Patch-level self-similarity heatmaps (DINOv3 fig. 3 style)

Every notebook so far has probed the single *pooled* vector each encoder produces per image. But UNI2-h / Virchow2 are ViTs -- internally, an image is a 16x16 grid of patch tokens (224px / 14px patch size) before any pooling happens. This notebook keeps that grid: pick one query patch in an image, compute its cosine similarity to every other patch token *in the same image*, and overlay the result as a heatmap -- the same figure DINOv2/DINOv3 papers use to show emergent part-level correspondence in self-supervised ViTs.

This needs the raw spatial tokens, which the saved `00_preparation/01_encodings` embeddings don't keep (they're already pooled to one vector), so this notebook loads the encoders itself via `vfm_encoders.embed_image_patches`.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from pathlib import Path

import cv2
import openslide
import torch
from nbhelper import (
    np,
    pd,
    plt,
)
from PIL import Image

from patch_similarity import (  # noqa: E402
    cosine_similarity_grid,
    denormalize,
    get_normalize_stats,
    upsample_heatmap,
)
from vfm_encoders import embed_image_patches, load_uni2, load_virchow2  # noqa: E402

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and [Virchow2](https://huggingface.co/paige-ai/Virchow2) model pages, then set `HF_TOKEN` (or leave unset for an interactive login prompt).

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load the encoders

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}

# Workaround: some deployments of the shared vfm_encoders.py predate the
# Encoder.patch_tokens field this notebook needs (embed_image_patches calls
# it to get spatial tokens instead of the pooled vector). Only patches it in
# if it's actually missing, so this becomes a no-op once the shared copy is
# updated -- see this repo's own src/vfm_encoders.py for the canonical version.
if not hasattr(encoders["uni2-h"], "patch_tokens"):

    def _uni2_patch_tokens(batch, model=encoders["uni2-h"].model):
        # no_embed_class + reg_tokens=8 -> 1 CLS + 8 register tokens prefix
        return model.forward_features(batch)[:, 9:]

    encoders["uni2-h"].patch_tokens = _uni2_patch_tokens

if not hasattr(encoders["virchow2"], "patch_tokens"):

    def _virchow2_patch_tokens(batch, model=encoders["virchow2"].model):
        # model(batch) already returns the full sequence; 1 CLS + 4 register tokens prefix
        return model(batch)[:, 5:]

    encoders["virchow2"].patch_tokens = _virchow2_patch_tokens

### 2. Pick example images

Two datasets, two very different fields of view:

- **CAMELYON17** ships as pre-cut 96x96px patches -- one tumor patch and one normal patch, so the heatmaps have a clear structure to key on (tumor cell nests vs. lymphoid tissue).
- **PANDA** ships as raw gigapixel WSIs instead of pre-cut patches, so a region can be cut at whatever size is useful for looking at it -- here a 448x448px foreground-tissue crop (read straight off level 0, so still native resolution), several times the field of view of a CAMELYON patch, from a high-grade and a benign slide.

In [ ]:
camelyon_base_dir = Path("/home/shared/data/camelyon17/camelyon17_v1.0/")

metadata_df = (
    pd.read_csv(camelyon_base_dir / "metadata.csv", index_col=False)
    .drop("Unnamed: 0", axis=1)
    .assign(
        patient_node=lambda df_: df_.apply(
            lambda row: f"patient_{row['patient']:03d}_node_{row['node']}", axis=1
        )
    )
    .assign(
        filepath_abs=lambda df_: df_.apply(
            lambda row: camelyon_base_dir
            / "patches"
            / row["patient_node"]
            / f"patch_{row['patient_node']}_x_{row['x_coord']}_y_{row['y_coord']}.png",
            axis=1,
        )
    )
)

sample_rows = pd.concat(
    [
        metadata_df[metadata_df["tumor"] == 1].sample(1, random_state=7),
        metadata_df[metadata_df["tumor"] == 0].sample(1, random_state=7),
    ]
)

images = {
    f"tumor={row.tumor} (center {row.center})": Image.open(row.filepath_abs).convert("RGB")
    for row in sample_rows.itertuples()
}

fig, axs = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
for ax, (title, img) in zip(axs, images.items()):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
panda_base_dir = Path("/home/shared/data/panda/")
panda_images_dir = panda_base_dir / "train_images"

panda_df = (
    pd.read_csv(panda_base_dir / "train.csv")
    .assign(filepath=lambda df_: df_["image_id"].apply(lambda y: panda_images_dir / f"{y}.tiff"))
    .loc[lambda df_: df_["filepath"].apply(lambda p: p.exists())]
)


def sample_foreground_region(
    slide_path, size=448, foreground_threshold=0.5, thumbnail_size=1024, rng=None
):
    """Picks a random size x size tile with >= foreground_threshold Otsu tissue
    coverage and reads it at level-0 (native) resolution -- same foreground-tiling
    approach as 00_preparation/01_encodings/encode_panda.ipynb, but for picking a
    single example region rather than exhaustively tiling the whole slide."""
    rng = rng or np.random.default_rng()
    slide = openslide.OpenSlide(str(slide_path))
    try:
        width, height = slide.dimensions
        thumb_arr = np.array(slide.get_thumbnail((thumbnail_size, thumbnail_size)).convert("L"))
        _, mask = cv2.threshold(thumb_arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8)) > 0
        scale_x, scale_y = width / thumb_arr.shape[1], height / thumb_arr.shape[0]

        candidates = []
        for y0 in range(0, height - size + 1, size):
            my0, my1 = int(y0 / scale_y), max(int(y0 / scale_y) + 1, int((y0 + size) / scale_y))
            for x0 in range(0, width - size + 1, size):
                mx0 = int(x0 / scale_x)
                mx1 = max(mx0 + 1, int((x0 + size) / scale_x))
                if mask[my0:my1, mx0:mx1].mean() >= foreground_threshold:
                    candidates.append((x0, y0))
        if not candidates:
            raise ValueError(
                f"No foreground tile >= {foreground_threshold} coverage in {slide_path}"
            )
        x0, y0 = candidates[rng.integers(len(candidates))]
        return slide.read_region((x0, y0), level=0, size=(size, size)).convert("RGB")
    finally:
        slide.close()


panda_sample_rows = pd.concat(
    [
        panda_df[panda_df["isup_grade"] >= 4].sample(1, random_state=7),
        panda_df[panda_df["isup_grade"] == 0].sample(1, random_state=7),
    ]
)

panda_rng = np.random.default_rng(7)
for row in panda_sample_rows.itertuples():
    images[f"PANDA ISUP {row.isup_grade} ({row.data_provider})"] = sample_foreground_region(
        row.filepath, rng=panda_rng
    )

fig, axs = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
for ax, (title, img) in zip(axs, images.items()):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 3. Self-similarity heatmap

For a query grid position `(row, col)`, embed the image into its patch-token grid, take that one token as the query vector, and show its cosine similarity to every patch in the same image. `(8, 8)` is roughly the center of the 16x16 grid -- change it below to probe a different spot.

In [ ]:
def show_similarity_grid(images, encoders, query_rc, cmap="viridis"):
    fig, axs = plt.subplots(
        len(images), 1 + len(encoders), figsize=(4 * (1 + len(encoders)), 4 * len(images))
    )
    if len(images) == 1:
        axs = axs[None, :]

    for row_idx, (title, image) in enumerate(images.items()):
        encoder0 = next(iter(encoders.values()))
        patch_size = encoder0.model.patch_embed.patch_size[0]
        mean, std = get_normalize_stats(encoder0.transform)
        display_img = denormalize(encoder0.transform(image), mean, std)
        qy = query_rc[0] * patch_size + patch_size // 2
        qx = query_rc[1] * patch_size + patch_size // 2

        ax = axs[row_idx, 0]
        ax.imshow(display_img)
        ax.scatter([qx], [qy], c="red", marker="*", s=200, edgecolors="white")
        ax.set_title(f"{title}\nquery patch")
        ax.axis("off")

        for col_idx, (model_name, encoder) in enumerate(encoders.items(), start=1):
            grid = embed_image_patches(encoder, image, device=device)  # (H, W, D)
            sim = cosine_similarity_grid(grid[query_rc], grid)
            heat = upsample_heatmap(sim, patch_size)

            ax = axs[row_idx, col_idx]
            ax.imshow(display_img)
            im = ax.imshow(heat, cmap=cmap, alpha=0.55, vmin=0, vmax=1)
            ax.scatter([qx], [qy], c="cyan", marker="*", s=200, edgecolors="black")
            ax.set_title(f"{model_name} -- cosine sim. to query")
            ax.axis("off")

    fig.colorbar(im, ax=axs, shrink=0.6, label="cosine similarity")
    return fig


show_similarity_grid(images, encoders, query_rc=(8, 8))
plt.show()

### 4. A different query patch

Move the query somewhere else in the grid -- e.g. `(3, 3)`, closer to a corner -- to see whether the heatmap still tracks a coherent structure (same cell type / tissue region) rather than just nearby pixels.

In [ ]:
show_similarity_grid(images, encoders, query_rc=(3, 3))
plt.show()

### Takeaways

- A heatmap that lights up other patches of the *same* structure (e.g. other tumor-cell-dense regions, or the same stroma texture) rather than just a blob around the query point is the DINO-style "emergent correspondence" signature -- evidence the encoder represents tissue type, not just local pixel statistics, at the patch level.
- Compare sharpness/locality between UNI2-h and Virchow2 -- one may produce tighter, more semantically bounded regions than the other.
- A query patch on background/normal tissue should produce a much more diffuse (or differently-shaped) heatmap than one on a tumor nest, if the encoder has learned to separate the two.
- The next notebook takes this cross-image: does a patch's nearest neighbor *in a different slide entirely* still look like the same tissue?